# 03 — Perceptron ve LightGBM

**Sahibi:** Öğrenci 2 · **Rubrik:** Model Engineering — Perceptron ve GBDT (20 puanın parçası)

| Bölüm | İçerik | Çıktı |
|---|---|---|
| 1 | Veri | — |
| 2 | Referans çizgisi: DummyClassifier | — |

In [1]:
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
import pandas as pd
from sklearn.dummy import DummyClassifier

from src.config import SEED
from src.data import load_split, to_xy
from src.metrics import evaluate
from src.plots import set_style

# Single switch for the training data: "clean", "smote" or "ctgan" (val/test are always clean)
VARIANT = "clean"

set_style()
pd.set_option("display.float_format", "{:.4f}".format)

## 1. Veri

Öğrenci 1'in temizlenmiş ve doldurulmuş verisi (`clean`). Model seçimi **val** setinde
yapılır; **test** seti en sona, final modellere ayrılmıştır ve bu bölümde yüklenmez.

In [3]:
train = load_split("train", VARIANT)
val = load_split("val", "clean")
X_train, y_train = to_xy(train)
X_val, y_val = to_xy(val)

pd.DataFrame(
    {
        "rows": [len(train), len(val)],
        "default_rate": [y_train.mean(), y_val.mean()],
    },
    index=[f"train ({VARIANT})", "val"],
)

,rows,default_rate
train (clean),20975,0.2213
val,4495,0.2211


## 2. Referans çizgisi: DummyClassifier

`DummyClassifier` özelliklere hiç bakmaz. Gerçek modellerin skorları ancak bu referansla
kıyaslanınca anlam kazanır.

- **most_frequent:** herkese çoğunluk sınıfını (0 = ödeyecek) söyler.
- **stratified:** sınıf oranına göre rastgele tahmin eder (%22 olasılıkla 1).

In [4]:
results = {}
for strategy in ("most_frequent", "stratified"):
    dummy = DummyClassifier(strategy=strategy, random_state=SEED).fit(X_train, y_train)
    # Dummy models have no meaningful score to tune a threshold on, so predict() is used as is
    y_score = dummy.predict_proba(X_val)[:, 1]
    results[f"dummy_{strategy}"] = evaluate(y_val, y_score, dummy.predict(X_val))

pd.DataFrame(results).T

,macro_f1,weighted_f1,roc_auc,pr_auc,accuracy
dummy_most_frequent,0.4378,0.6820,0.5000,0.2211,0.7789
dummy_stratified,0.5019,0.6573,0.5019,0.2218,0.6578


**Yorum**

- **Accuracy tuzağı:** `most_frequent` tek bir temerrüdü bile yakalamadan **%77,9 accuracy**
  alıyor. Macro-F1 ise 0,438: sınıf 0'ın F1'i ≈ 0,876, sınıf 1'in F1'i 0, ortalaması 0,438.
  Accuracy bu yüzden yalnızca bilgi amaçlı raporlanıyor.
- **Eşikten bağımsız metriklerin tabanı:** rastgele skorlarda ROC-AUC ≈ 0,50 ve PR-AUC ≈ val'deki
  pozitif oranı (0,221). PR eğrisinin taban çizgisi bu yüzden 0,22.
- **`stratified`** sınıf oranında rastgele tahminle macro-F1 ≈ 0,50 alıyor; accuracy'si ise
  çoğunluk tahmininden *düşük* (0,658). Accuracy'ye bakan biri yanlış modeli seçerdi.
- **Hedef:** Gerçek modellerin macro-F1'de 0,50'nin, PR-AUC'de 0,22'nin belirgin üstüne çıkması
  gerekiyor. Öğrenci 1'in sabit parametreli LightGBM'i val'de 0,707 / 0,550 almıştı
  (`reports/tstr_trtr.csv`), iyi bir sonucun ölçeği bu.